In [1]:
import pandas as pd
import numpy as np
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderUnavailable, GeocoderTimedOut, GeocoderServiceError

In [ ]:
# read data
attractions = pd.read_csv("dataset_crawler-google-places_2026-03-12_05-01-36-132.csv")

In [4]:
attractions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         69 non-null     object 
 1   totalScore    69 non-null     float64
 2   reviewsCount  69 non-null     int64  
 3   street        69 non-null     object 
 4   city          69 non-null     object 
 5   state         69 non-null     object 
 6   countryCode   69 non-null     object 
 7   website       50 non-null     object 
 8   phone         38 non-null     object 
 9   categories/0  69 non-null     object 
 10  categories/1  49 non-null     object 
 11  categories/2  28 non-null     object 
 12  categories/3  16 non-null     object 
 13  categories/4  9 non-null      object 
 14  categories/5  3 non-null      object 
 15  categories/6  1 non-null      object 
 16  url           69 non-null     object 
 17  categoryName  69 non-null     object 
dtypes: float64(1), int64(1), object(

In [8]:
# fill missing parts safely
for col in ["street", "city", "state"]:
    attractions[col] = attractions[col].fillna("").astype(str).str.strip()

# build address
attractions["address"] = (
    attractions["street"] + ", " +
    attractions["city"] + ", " +
    attractions["state"] + ", Japan"
).str.replace(r"\s+,", ",", regex=True).str.replace(r",\s*,", ",", regex=True).str.strip(", ")

# geolocator with larger timeout
geolocator = Nominatim(user_agent="glenn_tokyo_geocoder", timeout=10)

# rate limiter with retries
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=3,
    error_wait_seconds=5,
    swallow_exceptions=True
)

# geocode function
def safe_geocode(address):
    try:
        return geocode(address)
    except (GeocoderTimedOut, GeocoderUnavailable, GeocoderServiceError):
        return None
    except Exception:
        return None

# apply geocoding
attractions["location"] = attractions["address"].apply(safe_geocode)

# extract lat/lon
attractions["latitude"] = attractions["location"].apply(lambda loc: loc.latitude if loc else np.nan)
attractions["longitude"] = attractions["location"].apply(lambda loc: loc.longitude if loc else np.nan)

# inspect results
print(attractions[["address", "latitude", "longitude"]].head(20))
print("\nMissing coordinates:")
print(attractions[attractions["latitude"].isna()][["address"]])

RateLimiter caught an error, retrying (0/3 tries). Called with (*('4 Chome-2-8 Shibakoen, Minato, Tokyo, Japan',), **{}).
Traceback (most recent call last):
  File "c:\Users\Glenn\anaconda3\Lib\site-packages\geopy\geocoders\base.py", line 368, in _call_geocoder
    result = self.adapter.get_json(url, timeout=timeout, headers=req_headers)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Glenn\anaconda3\Lib\site-packages\geopy\adapters.py", line 472, in get_json
    resp = self._request(url, timeout=timeout, headers=headers)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Glenn\anaconda3\Lib\site-packages\geopy\adapters.py", line 500, in _request
    raise AdapterHTTPError(
geopy.adapters.AdapterHTTPError: Non-successful status code 429

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\Glenn\anaconda3\Lib\site-packages\geopy\extra\rate_li

                                              address   latitude   longitude
0         4 Chome-2-8 Shibakoen, Minato, Tokyo, Japan        NaN         NaN
1           1 Chome-1-2 Oshiage, Sumida, Tokyo, Japan        NaN         NaN
2            2 Chome-3-1 Asakusa, Taito, Tokyo, Japan        NaN         NaN
3                  1-1 Chiyoda, Chiyoda, Tokyo, Japan        NaN         NaN
4               11 Naitomachi, Shinjuku, Tokyo, Japan        NaN         NaN
5   2 Chome-8-1 Nishishinjuku, Shinjuku, Tokyo, Japan        NaN         NaN
6   Shibuya, 2 Chome−24−12 スクランブルスクエア 14階・45階 46階・...        NaN         NaN
7                  13-9 Uenokoen, Taito, Tokyo, Japan        NaN         NaN
8        2-1 Yoyogikamizonocho, Shibuya, Tokyo, Japan        NaN         NaN
9   Kabukicho, 1 Chome−1−6 あかるい花園 五番街, Shinjuku, T...        NaN         NaN
10    1 Chome-2 Nishishinjuku, Shinjuku, Tokyo, Japan  35.693069  139.699502
11          1 Chome-36-3 Asakusa, Taito, Tokyo, Japan  35.711950  139.794855

In [10]:
pd.set_option("display.max_rows", 10)
attractions[attractions['location'].isna()]

,title,totalScore,reviewsCount,street,city,state,countryCode,website,phone,categories/0,...,categories/3,categories/4,categories/5,categories/6,url,categoryName,address,location,latitude,longitude
0,Tokyo Tower,4.5,94176,4 Chome-2-8 Shibakoen,Minato,Tokyo,JP,https://www.tokyotower.co.jp/,+81 3-3433-5111,Observation deck,...,Shopping mall,Tourist attraction,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Observation deck,"4 Chome-2-8 Shibakoen, Minato, Tokyo, Japan",None,NaN,NaN
1,Tokyo Skytree,4.4,112070,1 Chome-1-2 Oshiage,Sumida,Tokyo,JP,https://www.tokyo-skytree.jp/?utm_source=googl...,+81 570-550-634,Observation deck,...,NaN,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Observation deck,"1 Chome-1-2 Oshiage, Sumida, Tokyo, Japan",None,NaN,NaN
2,Sensō-ji,4.5,92233,2 Chome-3-1 Asakusa,Taito,Tokyo,JP,https://www.senso-ji.jp/,+81 3-3842-0181,Buddhist temple,...,NaN,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Buddhist temple,"2 Chome-3-1 Asakusa, Taito, Tokyo, Japan",None,NaN,NaN
3,Imperial Palace East National Gardens,4.4,9639,1-1 Chiyoda,Chiyoda,Tokyo,JP,https://www.kunaicho.go.jp/event/higashigyoen/...,+81 3-3213-2050,National park,...,Park,Tourist attraction,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,National park,"1-1 Chiyoda, Chiyoda, Tokyo, Japan",None,NaN,NaN
4,Shinjuku Gyoen National Garden,4.6,43344,11 Naitomachi,Shinjuku,Tokyo,JP,https://www.env.go.jp/garden/shinjukugyoen/ind...,+81 3-3350-0151,Garden,...,Tourist attraction,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Garden,"11 Naitomachi, Shinjuku, Tokyo, Japan",None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,TOKYO TOWER TOURIST INFORMATION CENTER,4.5,24,4 Chome-2-8 Shibakoen,Minato,Tokyo,JP,NaN,+81 3-6435-4661,Tourist information center,...,NaN,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist information center,"4 Chome-2-8 Shibakoen, Minato, Tokyo, Japan",None,NaN,NaN
61,Yebisu Brewery Tokyo,4.3,4555,"Ebisu, 4 Chome−20−1 ガーデンプレイス 内",Shibuya,Tokyo,JP,http://www.sapporobeer.jp/brewery/y_museum/ind...,+81 3-5423-7255,Tourist attraction,...,Technology museum,Museum,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"Ebisu, 4 Chome−20−1 ガーデンプレイス 内, Shibuya, Tokyo...",None,NaN,NaN
63,Tokyo Virtual Circuit,4.0,46,5 Chome-4-6 Nishigotanda,Shinagawa,Tokyo,JP,http://sunakojukuchou.com/,+81 3-5962-7574,Tourist attraction,...,NaN,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"5 Chome-4-6 Nishigotanda, Shinagawa, Tokyo, Japan",None,NaN,NaN
66,Sky Bus Tokyo Marunouchi Ticket Counter,4.1,87,"Marunouchi, 2 Chome−5−2 Mitsubishi Building, １階",Chiyoda,Tokyo,JP,https://www.skybus.jp/?utm_source=skygmap&utm_...,+81 3-3215-0008,Tourist attraction,...,Event ticket seller,Ticket office,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"Marunouchi, 2 Chome−5−2 Mitsubishi Building, １...",None,NaN,NaN


In [11]:
attractions_address = attractions[["title", "state", "countryCode"]]
attractions_address['countryCode'] = np.where(attractions_address["countryCode"] == "JP", "Japan", attractions_address["countryCode"])
attractions_address['address'] = (attractions_address["title"] + "," + attractions_address["state"] + "," + attractions_address["countryCode"])

geolocator = Nominatim(user_agent="geo_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

attractions_address["location"] = attractions_address["address"].apply(geocode)
attractions_address["latitude"] = attractions_address["location"].apply(lambda loc: loc.latitude if loc else None)
attractions_address["longitude"] = attractions_address["location"].apply(lambda loc: loc.longitude if loc else None)

RateLimiter caught an error, retrying (0/2 tries). Called with (*('Shiokaze Park,Tokyo,Japan',), **{}).
Traceback (most recent call last):
  File "c:\Users\Glenn\anaconda3\Lib\site-packages\urllib3\connectionpool.py", line 537, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "c:\Users\Glenn\anaconda3\Lib\site-packages\urllib3\connection.py", line 461, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Glenn\anaconda3\Lib\http\client.py", line 1386, in getresponse
    response.begin()
  File "c:\Users\Glenn\anaconda3\Lib\http\client.py", line 325, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Glenn\anaconda3\Lib\http\client.py", line 286, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Glenn\anaconda3\Lib

In [15]:
cols = ["location", "latitude", "longitude"]

attractions[cols] = attractions[cols].combine_first(attractions_address[cols])

In [16]:
missing = attractions[attractions['location'].isna()]

In [17]:
missing

,title,totalScore,reviewsCount,street,city,state,countryCode,website,phone,categories/0,...,categories/3,categories/4,categories/5,categories/6,url,categoryName,address,location,latitude,longitude
3,Imperial Palace East National Gardens,4.4,9639,1-1 Chiyoda,Chiyoda,Tokyo,JP,https://www.kunaicho.go.jp/event/higashigyoen/...,+81 3-3213-2050,National park,...,Park,Tourist attraction,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,National park,"1-1 Chiyoda, Chiyoda, Tokyo, Japan",None,NaN,NaN
5,Tokyo Metropolitan Government Building No.1,4.5,6381,2 Chome-8-1 Nishishinjuku,Shinjuku,Tokyo,JP,https://www.metro.tokyo.lg.jp/,+81 3-5321-1111,Japanese prefecture government office,...,State government office,Tourist attraction,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Japanese prefecture government office,"2 Chome-8-1 Nishishinjuku, Shinjuku, Tokyo, Japan",None,NaN,NaN
19,Ninja Trick House In Tokyo,4.4,415,"Kabukicho, 2 Chome−28−13 第一和幸ビル 4階",Shinjuku,Tokyo,JP,https://ninja-trick-house.com/,+81 3-6457-3337,Amusement center,...,NaN,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Amusement center,"Kabukicho, 2 Chome−28−13 第一和幸ビル 4階, Shinjuku, ...",None,NaN,NaN
26,Maple Waterfall,4.2,178,4 Chome-3-25 Shibakoen,Minato,Tokyo,JP,NaN,NaN,Tourist attraction,...,NaN,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"4 Chome-3-25 Shibakoen, Minato, Tokyo, Japan",None,NaN,NaN
36,Tokyo Station Marunouchi South Gate Dome,4.6,53,"Marunouchi, 1 Chome−9 東京駅",Chiyoda,Tokyo,JP,http://www.tokyostationcity.com/learning/stati...,NaN,Tourist attraction,...,NaN,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"Marunouchi, 1 Chome−9 東京駅, Chiyoda, Tokyo, Japan",None,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,TOKYO TOWER TOURIST INFORMATION CENTER,4.5,24,4 Chome-2-8 Shibakoen,Minato,Tokyo,JP,NaN,+81 3-6435-4661,Tourist information center,...,NaN,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist information center,"4 Chome-2-8 Shibakoen, Minato, Tokyo, Japan",None,NaN,NaN
61,Yebisu Brewery Tokyo,4.3,4555,"Ebisu, 4 Chome−20−1 ガーデンプレイス 内",Shibuya,Tokyo,JP,http://www.sapporobeer.jp/brewery/y_museum/ind...,+81 3-5423-7255,Tourist attraction,...,Technology museum,Museum,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"Ebisu, 4 Chome−20−1 ガーデンプレイス 内, Shibuya, Tokyo...",None,NaN,NaN
63,Tokyo Virtual Circuit,4.0,46,5 Chome-4-6 Nishigotanda,Shinagawa,Tokyo,JP,http://sunakojukuchou.com/,+81 3-5962-7574,Tourist attraction,...,NaN,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"5 Chome-4-6 Nishigotanda, Shinagawa, Tokyo, Japan",None,NaN,NaN
66,Sky Bus Tokyo Marunouchi Ticket Counter,4.1,87,"Marunouchi, 2 Chome−5−2 Mitsubishi Building, １階",Chiyoda,Tokyo,JP,https://www.skybus.jp/?utm_source=skygmap&utm_...,+81 3-3215-0008,Tourist attraction,...,Event ticket seller,Ticket office,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"Marunouchi, 2 Chome−5−2 Mitsubishi Building, １...",None,NaN,NaN


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# example dataframe
df = pd.DataFrame({
    "street": ["4 Chome-2-8 Shibakoen",
               "1 Chome-1-2 Oshiage",
               "2 Chome-3-1 Asakusa"],
    "city": ["Minato", "Sumida", "Taito"],
    "state": ["Tokyo", "Tokyo", "Tokyo"],
    "countryCode": ["JP","JP","JP"]
})

import re

def convert_address(street):
    m = re.match(r"(\d+) Chome-(\d+)-(\d+) (.+)", street)
    if m:
        chome, block, building, area = m.groups()
        return f"{chome}-{block}-{building} {area}"
    return street

df["street_fixed"] = df["street"].apply(convert_address)

df["address"] = (
    df["street_fixed"] + ", " +
    df["city"] + ", " +
    df["state"] + ", Japan"
)

print(df["address"])

# geocode
geolocator = Nominatim(user_agent="geo_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

df["location"] = df["address"].apply(geocode)

df["latitude"] = df["location"].apply(lambda loc: loc.latitude if loc else None)
df["longitude"] = df["location"].apply(lambda loc: loc.longitude if loc else None)

print(df[["address","latitude","longitude"]])

0    4-2-8 Shibakoen, Minato, Tokyo, Japan
1      1-1-2 Oshiage, Sumida, Tokyo, Japan
2       2-3-1 Asakusa, Taito, Tokyo, Japan
Name: address, dtype: object
                                 address   latitude   longitude
0  4-2-8 Shibakoen, Minato, Tokyo, Japan  35.655145  139.751646
1    1-1-2 Oshiage, Sumida, Tokyo, Japan  35.712539  139.814815
2     2-3-1 Asakusa, Taito, Tokyo, Japan  35.717597  139.797563


In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# example dataframe
df = pd.DataFrame({
    "street": ["4 Chome-2-8 Shibakoen",
               "1 Chome-1-2 Oshiage",
               "2 Chome-3-1 Asakusa"],
    "city": ["Minato", "Sumida", "Taito"],
    "state": ["Tokyo", "Tokyo", "Tokyo"],
    "countryCode": ["JP","JP","JP"]
})

# build address
df["address"] = df["street"] + ", " + df["city"] + ", " + df["state"] + ", Japan"

# geocode
geolocator = Nominatim(user_agent="geo_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

df["location"] = df["address"].apply(geocode)

df["latitude"] = df["location"].apply(lambda loc: loc.latitude if loc else None)
df["longitude"] = df["location"].apply(lambda loc: loc.longitude if loc else None)

print(df[["address","latitude","longitude"]])

                                       address   latitude   longitude
0  4 Chome-2-8 Shibakoen, Minato, Tokyo, Japan        NaN         NaN
1    1 Chome-1-2 Oshiage, Sumida, Tokyo, Japan  35.710054  139.810714
2     2 Chome-3-1 Asakusa, Taito, Tokyo, Japan  35.713403  139.795526


In [ ]:
attractions_address.iloc[0:3,:]

,title,state,countryCode,address,location,latitude,longitude
0,Tokyo Tower,Tokyo,Japan,"Tokyo Tower,Tokyo,Japan","(東京タワー, 東京タワー通り, 芝公園四丁目, 芝公園, 港区, 東京都, 106-004...",35.658449,139.745536
1,Tokyo Skytree,Tokyo,Japan,"Tokyo Skytree,Tokyo,Japan","(東京スカイツリー, 2, 押上一丁目, 押上, 墨田区, 東京都, 131-0045, 日...",35.710054,139.810714
2,Sensō-ji,Tokyo,Japan,"Sensō-ji,Tokyo,Japan","(金龍山 浅草寺, 1, 浅草二丁目, 浅草, 台東区, 東京都, 111-0032, 日本...",35.713403,139.795526


In [ ]:
missing_locations = missing['address']
missing_idx = missing.index

3                    1-1 Chiyoda, Chiyoda, Tokyo, Japan
5     2 Chome-8-1 Nishishinjuku, Shinjuku, Tokyo, Japan
19    Kabukicho, 2 Chome−28−13 第一和幸ビル 4階, Shinjuku, ...
26         4 Chome-3-25 Shibakoen, Minato, Tokyo, Japan
36     Marunouchi, 1 Chome−9 東京駅, Chiyoda, Tokyo, Japan
                            ...                        
58          4 Chome-2-8 Shibakoen, Minato, Tokyo, Japan
61    Ebisu, 4 Chome−20−1 ガーデンプレイス 内, Shibuya, Tokyo...
63    5 Chome-4-6 Nishigotanda, Shinagawa, Tokyo, Japan
66    Marunouchi, 2 Chome−5−2 Mitsubishi Building, １...
67    Kaminarimon, 2 Chome−18−9 浅草文化観光センタ, Taito, To...
Name: address, Length: 14, dtype: object

In [34]:
import re

def clean_tokyo_address(addr):
    if pd.isna(addr):
        return None

    addr = str(addr).strip()

    # normalize dash
    addr = addr.replace("−", "-")

    # remove trailing ", Japan" first so we can add it back once only
    addr = re.sub(r",\s*Japan\s*$", "", addr)

    # split into comma parts
    parts = [p.strip() for p in addr.split(",") if p.strip()]

    # remove building / floor / station details from each part
    cleaned_parts = []
    for p in parts:
        # remove Japanese floor notation like 1階, 4階
        p = re.sub(r"\b\d+階\b", "", p)

        # remove common building / internal-place tails
        p = re.sub(r".*?ビル.*", "", p) if "ビル" in p else p
        p = re.sub(r".*?ガーデンプレイス.*", "", p) if "ガーデンプレイス" in p else p
        p = re.sub(r".*?Station.*", "", p, flags=re.IGNORECASE) if "Station" in p else p
        p = re.sub(r".*?Building.*", "", p, flags=re.IGNORECASE) if "Building" in p else p

        # remove some landmark text that appears after number block
        p = re.sub(r"\s+東京駅.*", "", p)
        p = re.sub(r"\s+浅草文化観光センタ.*", "", p)

        # normalize "2 Chome-28-13" -> "2-28-13"
        p = re.sub(r"(\d+)\s*Chome[-\s]*", r"\1-", p)

        # collapse whitespace
        p = re.sub(r"\s+", " ", p).strip()

        if p:
            cleaned_parts.append(p)

    parts = cleaned_parts

    # cases
    if len(parts) >= 4:
        # If first part is area name and second part starts with a number, merge them
        # Example: ["Kabukicho", "2-28-13", "Shinjuku", "Tokyo"]
        if re.match(r"^\d", parts[1]) and not re.match(r"^\d", parts[0]):
            first = f"{parts[0]} {parts[1]}"
            rest = parts[2:]
            final_parts = [first] + rest
        else:
            # keep first 4 parts only
            final_parts = parts[:4]

        return ", ".join(final_parts) + ", Japan"

    elif len(parts) == 3:
        return ", ".join(parts) + ", Japan"

    elif len(parts) == 2:
        return ", ".join(parts) + ", Japan"

    elif len(parts) == 1:
        return parts[0] + ", Japan"

    return addr

In [35]:
reformatted_addresses = {
    i: clean_tokyo_address(attractions.loc[i, "address"])
    for i in missing_idx
}

print(reformatted_addresses)

{3: '1-1 Chiyoda, Chiyoda, Tokyo, Japan', 5: '2-8-1 Nishishinjuku, Shinjuku, Tokyo, Japan', 19: 'Kabukicho, Shinjuku, Tokyo, Japan', 26: '4-3-25 Shibakoen, Minato, Tokyo, Japan', 36: 'Marunouchi 1-9, Chiyoda, Tokyo, Japan', 37: '1-1 Hamarikyuteien, Chuo, Tokyo, Japan', 40: '2-2 Yoyogikamizonocho, Shibuya, Tokyo, Japan', 50: 'Catetta Shiodome 1-8-2 Higashishinbashi, Minato, Tokyo, Japan', 55: 'Nishishinjuku, Shinjuku, Tokyo, Japan', 58: '4-2-8 Shibakoen, Minato, Tokyo, Japan', 61: 'Ebisu, Shibuya, Tokyo, Japan', 63: '5-4-6 Nishigotanda, Shinagawa, Tokyo, Japan', 66: 'Marunouchi, Chiyoda, Tokyo, Japan', 67: 'Kaminarimon 2-18-9, Taito, Tokyo, Japan'}


In [36]:
for idx, new_addr in reformatted_addresses.items():
    missing.loc[idx, "address"] = new_addr

In [37]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="glenn_tokyo_project")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

missing_idx = list(reformatted_addresses.keys())

missing.loc[missing_idx, "location"] = (
    missing.loc[missing_idx, "address"].apply(geocode)
)

missing.loc[missing_idx, "latitude"] = (
    missing.loc[missing_idx, "location"]
    .apply(lambda x: x.latitude if x else None)
)

missing.loc[missing_idx, "longitude"] = (
    missing.loc[missing_idx, "location"]
    .apply(lambda x: x.longitude if x else None)
)

In [41]:
failed = missing[missing['latitude'].isna()]

In [42]:
import pandas as pd
import numpy as np
import re
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="glenn_tokyo_geocoder", timeout=10)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.2,
    max_retries=3,
    error_wait_seconds=3,
    swallow_exceptions=True
)

def clean_address(addr):
    if pd.isna(addr):
        return None
    addr = str(addr).replace("−", "-").strip()
    addr = re.sub(r"\b\d+階\b", "", addr)              # remove floor
    addr = re.sub(r"第.*?ビル.*", "", addr)            # remove some building text
    addr = re.sub(r"\s+", " ", addr).strip(" ,")
    return addr

def geocode_with_fallback(row):
    queries = []

    # 1. original raw address
    if pd.notna(row.get("address")):
        queries.append(str(row["address"]).replace("−", "-").strip())

    # 2. cleaned address
    cleaned = clean_address(row.get("address"))
    if cleaned:
        queries.append(cleaned)

    # 3. title + city + state + country
    title = str(row.get("title", "")).strip()
    city = str(row.get("city", "")).strip()
    state = str(row.get("state", "")).strip()
    country = str(row.get("countryCode", "Japan")).strip()

    if title:
        queries.append(f"{title}, {city}, {state}, {country}")

    # 4. title + Tokyo + Japan
    if title:
        queries.append(f"{title}, Tokyo, Japan")

    # remove duplicates / empty
    seen = set()
    final_queries = []
    for q in queries:
        q = q.strip(" ,")
        if q and q not in seen:
            seen.add(q)
            final_queries.append(q)

    for q in final_queries:
        loc = geocode(q)
        if loc is not None:
            return loc, q

    return None, None

In [46]:
results = failed.apply(geocode_with_fallback, axis=1)

failed["location"] = results.apply(lambda x: x[0])
failed["query_used"] = results.apply(lambda x: x[1])

failed["latitude"] = failed["location"].apply(lambda loc: loc.latitude if loc else np.nan)
failed["longitude"] = failed["location"].apply(lambda loc: loc.longitude if loc else np.nan)

In [47]:
failed

,title,totalScore,reviewsCount,street,city,state,countryCode,website,phone,categories/0,...,categories/4,categories/5,categories/6,url,categoryName,address,location,latitude,longitude,query_used
26,Maple Waterfall,4.2,178,4 Chome-3-25 Shibakoen,Minato,Tokyo,JP,NaN,NaN,Tourist attraction,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"4-3-25 Shibakoen, Minato, Tokyo, Japan",None,NaN,NaN,None
37,300 Year-Old Pine,4.5,283,1-1 Hamarikyuteien,Chuo,Tokyo,JP,NaN,NaN,Tourist attraction,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"1-1 Hamarikyuteien, Chuo, Tokyo, Japan",None,NaN,NaN,None
40,First Flight Monument,3.8,40,2-2 Yoyogikamizonocho,Shibuya,Tokyo,JP,https://www.tokyo-park.or.jp/park/format/view0...,NaN,Tourist attraction,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"2-2 Yoyogikamizonocho, Shibuya, Tokyo, Japan",None,NaN,NaN,None
50,Ad Museum Tokyo,4.4,886,"Catetta Shiodome, 1 Chome-8-2 Higashishinbashi",Minato,Tokyo,JP,http://www.admt.jp/,+81 3-6218-2500,Tourist attraction,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"Catetta Shiodome 1-8-2 Higashishinbashi, Minat...",None,NaN,NaN,None
63,Tokyo Virtual Circuit,4.0,46,5 Chome-4-6 Nishigotanda,Shinagawa,Tokyo,JP,http://sunakojukuchou.com/,+81 3-5962-7574,Tourist attraction,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"5-4-6 Nishigotanda, Shinagawa, Tokyo, Japan","(東京バーチャルサーキット, 転坂, 赤坂六丁目, 赤坂, 港区, 東京都, 107-005...",35.669556,139.737035,"Tokyo Virtual Circuit, Tokyo, Japan"


In [55]:
cols = ["location", "latitude", "longitude"]

attractions[cols] = attractions[cols].combine_first(missing[cols])
attractions = attractions.dropna(subset=["location", "latitude", "longitude"]).reset_index(drop=True)

In [5]:
prices = [
    1500, 3100, 0, 0, 500, 800, 3400, 800, 0, 0,
    0, 0, 0, 0, 0, 300, 0, 0, 0, 4378,
    0, 0, 1500, 1500, 0, 0, 1600, 0, 0, 0,
    0, 0, 0, 0, 0, 0, 0, 500, 0, 0,
    0, 0, 0, 0, 0, 0, 1300, 0, 0, 4800,
    0, 0, 0, 0, 0, 0, 0, 0, 3300, 0,
    0, 2200, 0, 0
]
attractions["prices"] = prices

In [6]:
attractions

,Unnamed: 0,title,totalScore,reviewsCount,street,city,state,countryCode,website,phone,...,categories/4,categories/5,categories/6,url,categoryName,address,location,latitude,longitude,prices
0,0,Tokyo Tower,4.5,94176,4 Chome-2-8 Shibakoen,Minato,Tokyo,JP,https://www.tokyotower.co.jp/,+81 3-3433-5111,...,Tourist attraction,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Observation deck,"4 Chome-2-8 Shibakoen, Minato, Tokyo, Japan","東京タワー, 東京タワー通り, 芝公園四丁目, 芝公園, 港区, 東京都, 106-0041...",35.658449,139.745536,1500
1,1,Tokyo Skytree,4.4,112070,1 Chome-1-2 Oshiage,Sumida,Tokyo,JP,https://www.tokyo-skytree.jp/?utm_source=googl...,+81 570-550-634,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Observation deck,"1 Chome-1-2 Oshiage, Sumida, Tokyo, Japan","東京スカイツリー, 2, 押上一丁目, 押上, 墨田区, 東京都, 131-0045, 日本",35.710054,139.810714,3100
2,2,Sensō-ji,4.5,92233,2 Chome-3-1 Asakusa,Taito,Tokyo,JP,https://www.senso-ji.jp/,+81 3-3842-0181,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Buddhist temple,"2 Chome-3-1 Asakusa, Taito, Tokyo, Japan","金龍山 浅草寺, 1, 浅草二丁目, 浅草, 台東区, 東京都, 111-0032, 日本",35.713403,139.795526,0
3,3,Imperial Palace East National Gardens,4.4,9639,1-1 Chiyoda,Chiyoda,Tokyo,JP,https://www.kunaicho.go.jp/event/higashigyoen/...,+81 3-3213-2050,...,Tourist attraction,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,National park,"1-1 Chiyoda, Chiyoda, Tokyo, Japan","皇居, 1, 千代田, 千代田区, 東京都, 100-0001, 日本",35.683847,139.750686,0
4,4,Shinjuku Gyoen National Garden,4.6,43344,11 Naitomachi,Shinjuku,Tokyo,JP,https://www.env.go.jp/garden/shinjukugyoen/ind...,+81 3-3350-0151,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Garden,"11 Naitomachi, Shinjuku, Tokyo, Japan","新宿御苑, 内藤町, 新宿区, 東京都, 160-0014, 日本",35.685072,139.709547,500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,59,Maple Valley,4.3,282,4 Chome-3 Shibakoen,Minato,Tokyo,JP,https://www.kissport.or.jp/spot/tanbou/2111/,+81 3-3431-4359,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"4 Chome-3 Shibakoen, Minato, Tokyo, Japan","32芝公園ビル, 30, 芝公園三丁目, 芝公園, 港区, 東京都, 105-0011, 日本",35.658823,139.746420,0
60,60,Ginza Central Street,4.5,78,"Ginza, 3 Chome−6, 中央通り",Chuo,Tokyo,JP,NaN,NaN,...,NaN,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"Ginza, 3 Chome−6, 中央通り, Chuo, Tokyo, Japan","東銀座, 昭和通り, 銀座四丁目, 銀座, 中央区, 東京都, 104-0061, 日本",35.669819,139.767258,0
61,61,Sky Bus Tokyo Marunouchi Ticket Counter,4.1,87,"Marunouchi, 2 Chome−5−2 Mitsubishi Building, １階",Chiyoda,Tokyo,JP,https://www.skybus.jp/?utm_source=skygmap&utm_...,+81 3-3215-0008,...,Ticket office,NaN,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"Marunouchi, 2 Chome−5−2 Mitsubishi Building, １...","丸の内, 千代田区, 東京都, 100-0005, 日本",35.679070,139.765299,2200
62,62,Asakusa Culture Tourist Information Center,4.5,4258,"Kaminarimon, 2 Chome−18−9 浅草文化観光センタ",Taito,Tokyo,JP,https://www.city.taito.lg.jp/bunka_kanko/kanko...,+81 3-3842-5566,...,Wheelchair rental service,Wi-Fi spot,NaN,https://www.google.com/maps/search/?api=1&quer...,Tourist attraction,"Kaminarimon, 2 Chome−18−9 浅草文化観光センタ, Taito, To...","Observatoire d'Asakusa, 9, 雷門二丁目, 雷門, 台東区, 東京都...",35.710701,139.796551,0


In [2]:
attractions[["title", "prices"]]

,title,prices
0,Tokyo Tower,1500
1,Tokyo Skytree,3100
2,Sensō-ji,0
3,Imperial Palace East National Gardens,0
4,Shinjuku Gyoen National Garden,500
...,...,...
59,Maple Valley,0
60,Ginza Central Street,0
61,Sky Bus Tokyo Marunouchi Ticket Counter,2200
62,Asakusa Culture Tourist Information Center,0


In [ ]:
attractions.to_csv("attractions_with_coordinates_price.csv")